In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap
import os

# 1. LOAD DATA
# We load the Final Training Data
if not os.path.exists('data/processed/training_data_final.csv'):
    print("❌ Data not found. Run Notebook 02 first.")
    exit()

df = pd.read_csv('data/processed/training_data_final.csv')

# Take a larger sample for the map
df_sample = df.sample(n=3000, random_state=42)

print("🗺️ Generating Advanced Risk Dashboard...")

# 2. SETUP MAP
m = folium.Map(location=[22.0, 79.0], zoom_start=5, tiles='CartoDB dark_matter')
marker_cluster = MarkerCluster().add_to(m)

# 3. ADVANCED MARKER LOGIC
for _, row in df_sample.iterrows():
    # Calculate a simple "Risk Score" based on VPD (Scientific approximation)
    # High VPD (> 1.5) usually means High Fire Risk
    risk_score = row['vpd'] 
    
    # Determine Color based on Physics, not just the label
    if row['fire_detected'] == 1:
        color = '#ff0000' # Red (Actual Fire)
        status = "🔥 CONFIRMED FIRE"
    else:
        # For safe points, color them by how "close" they are to danger
        if row['vpd'] > 1.5:
            color = '#ffa500' # Orange (High Risk Environment)
            status = "⚠️ HIGH RISK (Safe but Dry)"
        elif row['vpd'] > 0.8:
            color = '#ffff00' # Yellow (Caution)
            status = "⚠️ CAUTION"
        else:
            color = '#00ff00' # Green (Safe)
            status = "🛡️ SAFE"

    # HTML Popup Content
    popup_html = f"""
    <b>{status}</b><br>
    Temp: {row['temp']:.1f}°C<br>
    Humidity: {row['humidity']:.1f}%<br>
    VPD: {row['vpd']:.2f} kPa<br>
    <i>(Higher VPD = Drier Air)</i>
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=200)
    ).add_to(marker_cluster)

# 4. ADD HEATMAP LAYER (For Density)
# Only show heatmap for Actual Fires
heat_data = df[df['fire_detected'] == 1][['latitude', 'longitude']].values.tolist()
HeatMap(heat_data, radius=15, blur=20, gradient={0.4: 'orange', 1: 'red'}).add_to(m)

# 5. SAVE
save_path = 'data/raw/advanced_risk_dashboard.html'
m.save(save_path)
print(f"✅ Dashboard saved to: {save_path}")
print("👉 Open this file in your browser to interact.")

🗺️ Generating Advanced Risk Dashboard...
✅ Dashboard saved to: data/raw/advanced_risk_dashboard.html
👉 Open this file in your browser to interact.
